## QFT Resource Estimation

This notebook contains all the code from the blog post *"Treating FTQC Like a Real System: A Practical Look at Quantum Resource Estimation"*.

It walks through the full QRE pipeline for Quantum Fourier Transform from logical gate counts, through QREF export and Bartiq symbolic compilation, to a surface-code physical cost model.

In [ ]:
import yaml
import numpy as np
import matplotlib.pyplot as plt

from qref import SchemaV1
from bartiq import compile_routine, evaluate

from qualtran.bloqs.qft.qft_text_book import QFTTextBook
from qualtran.bloqs.qft.approximate_qft import ApproximateQFT
from qualtran.resource_counting import get_cost_value, QECGatesCost, QubitCount
from qualtran.surface_code import (
    AlgorithmSummary,
    PhysicalCostModel,
    PhysicalParameters,
    QECScheme,
    CCZ2TFactory,
    SimpleDataBlock,
)

### 1. Define Primitive and Extract Logical Costs

In Qualtran a primitive is a *bloq*: a composable building block with typed quantum registers. `QECGatesCost()` walks the decomposition tree and tallies the gates that matter under surface-code QEC. Qualtran reports And/CCZ gates separately — each is equivalent to 4 T-gates, so the standard metric is `n_t + 4 x n_ccz` (the "T-equivalent count").

In [ ]:
# Defining the primitive
qft = QFTTextBook(bitsize=32)

# Extracting logical cost
cost = get_cost_value(qft, QECGatesCost())

print("QFT(32) logical costs:")
print(cost)
print()
print("Breakdown:")
print(f"  T-gates:    {cost.t}")
print(f"  And (≈CCZ): {cost.and_bloq}")
print(f"  Rotations:  {cost.rotation}")
print(f"  Cliffords:  {cost.clifford}")
print(f"  Measure:    {cost.measurement}")

In [ ]:
# Qubit count
n_qubits = get_cost_value(qft, QubitCount())
print(f"QFT(32) qubit count: {n_qubits}")

### 2. Verify Small Instance

Before trusting resource counts at scale, check correctness on parameters small enough for a classical simulator. Same pattern as classical software: unit-test the small case, trust the scaling analysis for the large one.

In [ ]:
# Verify small instances
small_qft = QFTTextBook(bitsize=3)

# Tensor contraction - get the unitary matrix
uni = small_qft.tensor_contract()
N = 2**3
print(f"Unitary shape: {uni.shape}")
print(f"Is unitary: {np.allclose(uni @ uni.conj().T, np.eye(N))}")

# Cirq circuit
composite = small_qft.decompose_bloq()
circuit = composite.to_cirq_circuit()
print(f"Cirq circuit depth: {len(circuit)}")
print(f"\nCircuit:\n{circuit}")

### 3. QFT Scaling Sweep

Sweep over bitsizes, collect T-equivalent counts, and qubits.

The T-count grows due to rotation synthesis.Comparing textbook QFT against an approximate variant that truncates small-angle rotations shows a dramatic T-count difference, and that difference cascades all the way down through the physical layer.

In [ ]:
bitsizes = [8, 16, 32, 64, 128]
results = []

for n in bitsizes:
    # Textbook QFT
    cost_tb = get_cost_value(QFTTextBook(bitsize=n), QECGatesCost())
    tb_counts = cost_tb.total_t_and_ccz_count()  # returns {'n_t': ..., 'n_ccz': ...}
    tb_total = tb_counts["n_t"] + 4 * tb_counts["n_ccz"]
    tb_qubits = get_cost_value(QFTTextBook(bitsize=n), QubitCount())

    # Approximate QFT (phase_bitsize = n//2)
    cost_ap = get_cost_value(
        ApproximateQFT(bitsize=n, phase_bitsize=n // 2), QECGatesCost()
    )
    ap_counts = cost_ap.total_t_and_ccz_count()
    ap_total = ap_counts["n_t"] + 4 * ap_counts["n_ccz"]

    results.append(
        {
            "n": n,
            "tb_t": tb_counts["n_t"],
            "tb_ccz": tb_counts["n_ccz"],
            "tb_total": tb_total,
            "tb_rotations": cost_tb.rotation,
            "tb_qubits": tb_qubits,
            "ap_t": ap_counts["n_t"],
            "ap_ccz": ap_counts["n_ccz"],
            "ap_total": ap_total,
        }
    )

    ratio = tb_total / ap_total if ap_total > 0 else float("inf")
    print(
        f"n={n:>4}  TB: {tb_total:>6} T-equiv  AP: {ap_total:>6} T-equiv  ratio: {ratio:.1f}x"
    )

In [ ]:
# Textbook vs Approximate QFT: T-equivalent count
ns = [r["n"] for r in results]
tb_totals = [r["tb_total"] for r in results]
ap_totals = [r["ap_total"] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: T-equivalent count — log scale to show superlinear growth clearly
ax1.semilogy(
    ns,
    tb_totals,
    "o-",
    color="#e74c3c",
    linewidth=3,
    markersize=10,
    label="Textbook QFT",
)
ax1.semilogy(
    ns,
    ap_totals,
    "s-",
    color="#3498db",
    linewidth=3,
    markersize=10,
    label="Approximate QFT (phase = n/2)",
)
ax1.set_xlabel("Bitsize (n)", fontsize=13)
ax1.set_ylabel("T-equivalent count  (n_t + 4·n_ccz)", fontsize=13)
ax1.set_title("Logical Non-Clifford Cost (log scale)", fontsize=14)
ax1.legend(fontsize=11, loc="lower right")
ax1.grid(True, alpha=0.3, which="both")
ax1.tick_params(axis="both", labelsize=11)

# Right: Cost ratio — how much the approximation saves
ratios = [tb / ap if ap > 0 else 0 for tb, ap in zip(tb_totals, ap_totals)]
valid_ns = [n for n, r in zip(ns, ratios) if r > 0]
valid_ratios = [r for r in ratios if r > 0]
ax2.plot(valid_ns, valid_ratios, "o-", color="#2ecc71", linewidth=3, markersize=10)
ax2.set_xlabel("Bitsize (n)", fontsize=13)
ax2.set_ylabel("Textbook / Approximate", fontsize=13)
ax2.set_title("T-equivalent Savings from Approximation", fontsize=14)
ax2.axhline(
    y=1, color="gray", linestyle="--", alpha=0.5, label="No savings (ratio = 1)"
)
ax2.legend(fontsize=12, loc="upper right")
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis="both", labelsize=10)
# Annotate each point with the ratio value
for x, y in zip(valid_ns, valid_ratios):
    ax2.annotate(
        f"{y:.1f}x",
        (x, y),
        textcoords="offset points",
        xytext=(0, -20),
        ha="center",
        fontsize=13,
        color="#2c3e50",
    )

plt.tight_layout()
plt.savefig("outputs/chart_qft_textbook_vs_approx.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cost breakdown: where do the T-equivalents come from?
# CCZ/And gates dominate; raw T-gates are a small fraction.
fig, ax = plt.subplots(figsize=(10, 5))

tb_ts = [r['tb_t'] for r in results]
tb_ccz4 = [4 * r['tb_ccz'] for r in results]  # each CCZ = 4 T-gates

x = range(len(ns))
bars_ccz = ax.bar(x, tb_ccz4, label='4 x CCZ (And bloqs)', color='#e74c3c', alpha=0.85)
bars_t = ax.bar(x, tb_ts, bottom=tb_ccz4, label='T-gates (raw)', color='#f39c12', alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels([str(n) for n in ns])
ax.set_xlabel('Bitsize (n)', fontsize=12)
ax.set_ylabel('T-equivalent count', fontsize=12)
ax.set_title('Textbook QFT: Non-Clifford Cost Breakdown', fontsize=13)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='both', labelsize=10)

# Label total on top of each bar
for i, (ccz, t) in enumerate(zip(tb_ccz4, tb_ts)):
    total = ccz + t
    ax.text(i, total + total * 0.02, f'{total:,}', ha='center', va='bottom',
            fontsize=9, color='#2c3e50', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/chart_qft_cost_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

### 4. Exporting to QREF

Qualtran gives us the concrete numbers. But what if we want to explore cost scaling without re-running decompositions every time? This is where a different part of the ecosystem comes in.

[QREF](https://github.com/PsiQ/qref) is PsiQuantum's open format for representing FTQC algorithms as hierarchical cost DAGs. [Bartiq](https://github.com/PsiQ/bartiq) propagates symbolic cost expressions through them. The two tools are entirely independent of Qualtran. We simply take the costs extracted above and package them into a format that Bartiq understands.

Here we build a concrete QREF program from the QFT(32) costs.

In [ ]:
# Build a QREF program from the QFT(32) costs we already have
qft32_cost = get_cost_value(QFTTextBook(bitsize=32), QECGatesCost())
qft32_qubits = get_cost_value(QFTTextBook(bitsize=32), QubitCount())
qft32_t_equiv = qft32_cost.t + 4 * qft32_cost.and_bloq

program = SchemaV1(
    version="v1",
    program={
        "name": "qft_textbook",
        "ports": [
            {"name": "in", "direction": "input", "size": 32},
            {"name": "out", "direction": "output", "size": 32},
        ],
        "resources": [
            {"name": "T_gates", "type": "additive", "value": qft32_t_equiv},
            {"name": "rotations", "type": "additive", "value": qft32_cost.rotation},
            {"name": "cliffords", "type": "additive", "value": qft32_cost.clifford},
            {"name": "n_qubits", "type": "additive", "value": int(qft32_qubits)},
        ],
        "children": [],
        "connections": [],
    },
)

# Save as YAML
qref_path = "outputs/qft_textbook.qref.yaml"
with open(qref_path, "w") as f:
    yaml.dump(program.model_dump(), f, default_flow_style=False)

print(f"QREF program saved to {qref_path}")
print()
print(yaml.dump(program.model_dump(), default_flow_style=False))

### 5. Symbolic Cost Propagation with Bartiq

Bartiq compiles a QREF program and lets you evaluate it. First, we compile the concrete QREF from above to verify the round-trip. Then we build a **parametric** version with a symbolic cost formula to demonstrate the what-if workflow.

The parametric formula (`n*(n-1)/2 * 13`) is an approximation, the exact T-cost depends on rotation-synthesis internals that only Qualtran knows, so the numbers will not match exactly. The point here is the mechanism: define costs as symbolic expressions, compile once, evaluate for many parameter values.

Where Bartiq really shines is in **hierarchical** algorithms: an outer routine (e.g. QPE) calling sub-routines (QFT, controlled-U) whose costs are symbolic. Bartiq propagates those expressions from leaves to root, so you change a parameter once and get updated costs for the whole algorithm.

In [ ]:
# Compile the concrete QREF program
compilation_result = compile_routine(program)
routine = compilation_result.routine

print("Compiled routine resources (concrete, should match section 1):")
for name, res in routine.resources.items():
    print(f"  {name}: {res.value}")

In [ ]:
# Parametric version with symbolic cost formula, evaluate for different n.
# The formula is approximate; the point is the workflow.

parametric_program = SchemaV1(
    version="v1",
    program={
        "name": "qft_parametric",
        "ports": [
            {"name": "in", "direction": "input", "size": "n"},
            {"name": "out", "direction": "output", "size": "n"},
        ],
        "input_params": ["n"],
        "resources": [
            {"name": "T_gates", "type": "additive", "value": "n*(n-1)/2 * 13"},
            {"name": "n_qubits", "type": "additive", "value": "n + 1"},
        ],
        "children": [],
        "connections": [],
    },
)

compilation_result_p = compile_routine(parametric_program)
routine_p = compilation_result_p.routine

print("Parametric evaluation (approximate formula):")
for n_val in [16, 32, 64, 128]:
    eval_result = evaluate(routine_p, assignments={"n": n_val})
    resources = eval_result.routine.resources
    print(
        f"  n={n_val:>4}  T_gates={resources['T_gates'].value:>8}  n_qubits={resources['n_qubits'].value}"
    )

### 6. Connecting to the Physical Layer

The logical costs become inputs to a physical resource model. Qualtran ships a built-in surface-code cost model: `AlgorithmSummary.from_bloq()` extracts costs from a bloq, and `PhysicalCostModel` wraps the hardware assumptions.

At this point you have the full vertical cut through the stack: algorithm intent => logical gate counts => physical hardware requirements.

In [ ]:
qft32 = QFTTextBook(bitsize=32)
summary = AlgorithmSummary.from_bloq(qft32)

print("Algorithm summary:")
print(f"  Logical gates    : {summary.n_logical_gates}")
print(f"  Algo qubits      : {summary.n_algo_qubits}")
# Note: from_bloq may report rotation_layers as None for some bloqs.
# The physical model still works as it uses n_logical_gates directly.
print(f"  Rotation layers  : {summary.n_rotation_layers}")

# Physical-layer assumptions
phys = PhysicalParameters(physical_error=1e-3, cycle_time_us=1.0)
qec = QECScheme(error_rate_scaler=0.03, error_rate_threshold=0.01)
factory = CCZ2TFactory()
data_block = SimpleDataBlock(data_d=17, routing_overhead=0.5)

model = PhysicalCostModel(
    physical_params=phys,
    data_block=data_block,
    factory=factory,
    qec_scheme=qec,
)

print()
print("Physical cost estimate (surface-code, d=17, p_err=1e-3):")
print(f"  Physical qubits  : {model.n_phys_qubits(summary):,}")
print(f"  Runtime (hours)  : {model.duration_hr(summary):.4f}")
print(f"  Cycles           : {model.n_cycles(summary):,}")
print(f"  Error            : {model.error(summary):.2e}")